In [1]:
#ortools求解


import chardet
import time
import datetime
import random
import collections
import plotly as py
import plotly.figure_factory as ff
# Import Python wrapper for or-tools CP-SAT solver.
from ortools.sat.python import cp_model
from ortools.linear_solver import pywraplp  #线性求解


def load_text(file_name):
    try:
        with open(file_name, "rb") as f:
            f_read = f.read()
            f_cha_info = chardet.detect(f_read)
            final_data = f_read.decode(f_cha_info['encoding'])
            return final_data, True
    except FileNotFoundError:
        return str(None), False


def rgb():
    return random.randint(0, 256)


# print(data)
def MinimalJobshopSat(string):
    a = list(map(int, string.split()))
    n, machines_count = a[0], a[1]
    all_machines = range(machines_count)
    jobs_data = []
    job = []
    for i, (j, k) in enumerate(zip(a[2::2], a[3::2])):
        job.append((j, k))
        if (i + 1) % machines_count == 0:
            jobs_data.append(job)
            job = []
    """Minimal jobshop problem."""
    # Create the model.
    model = cp_model.CpModel()

    # Computes horizon dynamically as the sum of all durations.
    horizon = sum(task[1] for job in jobs_data for task in job)

    # Named tuple to store information about created variables.
    task_type = collections.namedtuple('task_type', 'start end interval')
    # Named tuple to manipulate solution information.
    assigned_task_type = collections.namedtuple('assigned_task_type',
                                                'start job index duration')

    # Creates job intervals and add to the corresponding machine lists.
    all_tasks = {}
    machine_to_intervals = collections.defaultdict(list)

    for job_id, job in enumerate(jobs_data):
        for task_id, task in enumerate(job):
            machine = task[0]
            duration = task[1]
            suffix = '_%i_%i' % (job_id, task_id)
            start_var = model.NewIntVar(0, horizon, 'start' + suffix)
            end_var = model.NewIntVar(0, horizon, 'end' + suffix)
            interval_var = model.NewIntervalVar(start_var, duration, end_var,
                                                'interval' + suffix)
            all_tasks[job_id, task_id] = task_type(
                start=start_var, end=end_var, interval=interval_var)
            machine_to_intervals[machine].append(interval_var)

    # Create and add disjunctive constraints.
    for machine in all_machines:
        model.AddNoOverlap(machine_to_intervals[machine])

    # Precedences inside a job.
    for job_id, job in enumerate(jobs_data):
        for task_id in range(len(job) - 1):
            model.Add(all_tasks[job_id, task_id +
                                1].start >= all_tasks[job_id, task_id].end)

    # Makespan objective.
    obj_var = model.NewIntVar(0, horizon, 'makespan')
    model.AddMaxEquality(obj_var, [
        all_tasks[job_id, len(job) - 1].end
        for job_id, job in enumerate(jobs_data)
    ])
    model.Minimize(obj_var)

    # Solve model.
    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL:
        # Create one list of assigned tasks per machine.
        assigned_jobs = collections.defaultdict(list)
        for job_id, job in enumerate(jobs_data):
            for task_id, task in enumerate(job):
                machine = task[0]
                assigned_jobs[machine].append(
                    assigned_task_type(
                        start=solver.Value(all_tasks[job_id, task_id].start),
                        job=job_id,
                        index=task_id,
                        duration=task[1]))

        # Create per machine output lines.
        print('User time: %.2fs' % solver.UserTime())
        print('Wall time: %.2fs' % solver.WallTime())
        print('Optimal Schedule Length: %i' % solver.ObjectiveValue())

        # Draw GanttChart
        df = []
        today = dt.today()
        start_date = dt(today.year, today.month, today.day)
        for machine in all_machines[::-1]:
            # Sort by starting time.
            assigned_jobs[machine].sort()
            for assigned_task in assigned_jobs[machine]:
                # name = 'job_%i_%i' % (assigned_task.job, assigned_task.index)
                start = assigned_task.start
                duration = assigned_task.duration
                df.append(dict(Task="M%s" % (machine + 1), Start=start_date + time_delta(0, start),
                               Finish=start_date + time_delta(0, start + duration),
                               Resource="%s" % (assigned_task.job + 1), complete=assigned_task.job + 1))
        colors = {}
        for i in range(n):
            key = "%s" % (i + 1)
            colors[key] = "rgb(%s, %s, %s)" % (rgb(), rgb(), rgb())
        fig = ff.create_gantt(df, colors=colors, index_col='Resource', group_tasks=True, show_colorbar=True)
        pyplt(fig, filename=r"./GanttChart.html", auto_open=False)

def lp_solver():
    # Commodity, Unit, 1939 price (cents), Calories, Protein (g), Calcium (g), Iron (mg),
    # Vitamin A (IU), Thiamine (mg), Riboflavin (mg), Niacin (mg), Ascorbic Acid (mg)
    data = [
        ['Wheat Flour (Enriched)', '10 lb.', 36, 44.7, 1411, 2, 365, 0, 55.4, 33.3, 441, 0],
        ['Macaroni', '1 lb.', 14.1, 11.6, 418, 0.7, 54, 0, 3.2, 1.9, 68, 0],
        ['Wheat Cereal (Enriched)', '28 oz.', 24.2, 11.8, 377, 14.4, 175, 0, 14.4, 8.8, 114, 0],
        ['Corn Flakes', '8 oz.', 7.1, 11.4, 252, 0.1, 56, 0, 13.5, 2.3, 68, 0],
        ['Corn Meal', '1 lb.', 4.6, 36.0, 897, 1.7, 99, 30.9, 17.4, 7.9, 106, 0],
        ['Hominy Grits', '24 oz.', 8.5, 28.6, 680, 0.8, 80, 0, 10.6, 1.6, 110, 0],
        ['Rice', '1 lb.', 7.5, 21.2, 460, 0.6, 41, 0, 2, 4.8, 60, 0],
        ['Rolled Oats', '1 lb.', 7.1, 25.3, 907, 5.1, 341, 0, 37.1, 8.9, 64, 0],
        ['White Bread (Enriched)', '1 lb.', 7.9, 15.0, 488, 2.5, 115, 0, 13.8, 8.5, 126, 0],
        ['Whole Wheat Bread', '1 lb.', 9.1, 12.2, 484, 2.7, 125, 0, 13.9, 6.4, 160, 0],
        ['Rye Bread', '1 lb.', 9.1, 12.4, 439, 1.1, 82, 0, 9.9, 3, 66, 0],
        ['Pound Cake', '1 lb.', 24.8, 8.0, 130, 0.4, 31, 18.9, 2.8, 3, 17, 0],
        ['Soda Crackers', '1 lb.', 15.1, 12.5, 288, 0.5, 50, 0, 0, 0, 0, 0],
        ['Milk', '1 qt.', 11, 6.1, 310, 10.5, 18, 16.8, 4, 16, 7, 177],
        ['Evaporated Milk (can)', '14.5 oz.', 6.7, 8.4, 422, 15.1, 9, 26, 3, 23.5, 11, 60],
        ['Butter', '1 lb.', 30.8, 10.8, 9, 0.2, 3, 44.2, 0, 0.2, 2, 0],
        ['Oleomargarine', '1 lb.', 16.1, 20.6, 17, 0.6, 6, 55.8, 0.2, 0, 0, 0],
        ['Eggs', '1 doz.', 32.6, 2.9, 238, 1.0, 52, 18.6, 2.8, 6.5, 1, 0],
        ['Cheese (Cheddar)', '1 lb.', 24.2, 7.4, 448, 16.4, 19, 28.1, 0.8, 10.3, 4, 0],
        ['Cream', '1/2 pt.', 14.1, 3.5, 49, 1.7, 3, 16.9, 0.6, 2.5, 0, 17],
        ['Peanut Butter', '1 lb.', 17.9, 15.7, 661, 1.0, 48, 0, 9.6, 8.1, 471, 0],
        ['Mayonnaise', '1/2 pt.', 16.7, 8.6, 18, 0.2, 8, 2.7, 0.4, 0.5, 0, 0],
        ['Crisco', '1 lb.', 20.3, 20.1, 0, 0, 0, 0, 0, 0, 0, 0],
        ['Lard', '1 lb.', 9.8, 41.7, 0, 0, 0, 0.2, 0, 0.5, 5, 0],
        ['Sirloin Steak', '1 lb.', 39.6, 2.9, 166, 0.1, 34, 0.2, 2.1, 2.9, 69, 0],
        ['Round Steak', '1 lb.', 36.4, 2.2, 214, 0.1, 32, 0.4, 2.5, 2.4, 87, 0],
        ['Rib Roast', '1 lb.', 29.2, 3.4, 213, 0.1, 33, 0, 0, 2, 0, 0],
        ['Chuck Roast', '1 lb.', 22.6, 3.6, 309, 0.2, 46, 0.4, 1, 4, 120, 0],
        ['Plate', '1 lb.', 14.6, 8.5, 404, 0.2, 62, 0, 0.9, 0, 0, 0],
        ['Liver (Beef)', '1 lb.', 26.8, 2.2, 333, 0.2, 139, 169.2, 6.4, 50.8, 316, 525],
        ['Leg of Lamb', '1 lb.', 27.6, 3.1, 245, 0.1, 20, 0, 2.8, 3.9, 86, 0],
        ['Lamb Chops (Rib)', '1 lb.', 36.6, 3.3, 140, 0.1, 15, 0, 1.7, 2.7, 54, 0],
        ['Pork Chops', '1 lb.', 30.7, 3.5, 196, 0.2, 30, 0, 17.4, 2.7, 60, 0],
        ['Pork Loin Roast', '1 lb.', 24.2, 4.4, 249, 0.3, 37, 0, 18.2, 3.6, 79, 0],
        ['Bacon', '1 lb.', 25.6, 10.4, 152, 0.2, 23, 0, 1.8, 1.8, 71, 0],
        ['Ham, smoked', '1 lb.', 27.4, 6.7, 212, 0.2, 31, 0, 9.9, 3.3, 50, 0],
        ['Salt Pork', '1 lb.', 16, 18.8, 164, 0.1, 26, 0, 1.4, 1.8, 0, 0],
        ['Roasting Chicken', '1 lb.', 30.3, 1.8, 184, 0.1, 30, 0.1, 0.9, 1.8, 68, 46],
        ['Veal Cutlets', '1 lb.', 42.3, 1.7, 156, 0.1, 24, 0, 1.4, 2.4, 57, 0],
        ['Salmon, Pink (can)', '16 oz.', 13, 5.8, 705, 6.8, 45, 3.5, 1, 4.9, 209, 0],
        ['Apples', '1 lb.', 4.4, 5.8, 27, 0.5, 36, 7.3, 3.6, 2.7, 5, 544],
        ['Bananas', '1 lb.', 6.1, 4.9, 60, 0.4, 30, 17.4, 2.5, 3.5, 28, 498],
        ['Lemons', '1 doz.', 26, 1.0, 21, 0.5, 14, 0, 0.5, 0, 4, 952],
        ['Oranges', '1 doz.', 30.9, 2.2, 40, 1.1, 18, 11.1, 3.6, 1.3, 10, 1998],
        ['Green Beans', '1 lb.', 7.1, 2.4, 138, 3.7, 80, 69, 4.3, 5.8, 37, 862],
        ['Cabbage', '1 lb.', 3.7, 2.6, 125, 4.0, 36, 7.2, 9, 4.5, 26, 5369],
        ['Carrots', '1 bunch', 4.7, 2.7, 73, 2.8, 43, 188.5, 6.1, 4.3, 89, 608],
        ['Celery', '1 stalk', 7.3, 0.9, 51, 3.0, 23, 0.9, 1.4, 1.4, 9, 313],
        ['Lettuce', '1 head', 8.2, 0.4, 27, 1.1, 22, 112.4, 1.8, 3.4, 11, 449],
        ['Onions', '1 lb.', 3.6, 5.8, 166, 3.8, 59, 16.6, 4.7, 5.9, 21, 1184],
        ['Potatoes', '15 lb.', 34, 14.3, 336, 1.8, 118, 6.7, 29.4, 7.1, 198, 2522],
        ['Spinach', '1 lb.', 8.1, 1.1, 106, 0, 138, 918.4, 5.7, 13.8, 33, 2755],
        ['Sweet Potatoes', '1 lb.', 5.1, 9.6, 138, 2.7, 54, 290.7, 8.4, 5.4, 83, 1912],
        ['Peaches (can)', 'No. 2 1/2', 16.8, 3.7, 20, 0.4, 10, 21.5, 0.5, 1, 31, 196],
        ['Pears (can)', 'No. 2 1/2', 20.4, 3.0, 8, 0.3, 8, 0.8, 0.8, 0.8, 5, 81],
        ['Pineapple (can)', 'No. 2 1/2', 21.3, 2.4, 16, 0.4, 8, 2, 2.8, 0.8, 7, 399],
        ['Asparagus (can)', 'No. 2', 27.7, 0.4, 33, 0.3, 12, 16.3, 1.4, 2.1, 17, 272],
        ['Green Beans (can)', 'No. 2', 10, 1.0, 54, 2, 65, 53.9, 1.6, 4.3, 32, 431],
        ['Pork and Beans (can)', '16 oz.', 7.1, 7.5, 364, 4, 134, 3.5, 8.3, 7.7, 56, 0],
        ['Corn (can)', 'No. 2', 10.4, 5.2, 136, 0.2, 16, 12, 1.6, 2.7, 42, 218],
        ['Peas (can)', 'No. 2', 13.8, 2.3, 136, 0.6, 45, 34.9, 4.9, 2.5, 37, 370],
        ['Tomatoes (can)', 'No. 2', 8.6, 1.3, 63, 0.7, 38, 53.2, 3.4, 2.5, 36, 1253],
        ['Tomato Soup (can)', '10 1/2 oz.', 7.6, 1.6, 71, 0.6, 43, 57.9, 3.5, 2.4, 67, 862],
        ['Peaches, Dried', '1 lb.', 15.7, 8.5, 87, 1.7, 173, 86.8, 1.2, 4.3, 55, 57],
        ['Prunes, Dried', '1 lb.', 9, 12.8, 99, 2.5, 154, 85.7, 3.9, 4.3, 65, 257],
        ['Raisins, Dried', '15 oz.', 9.4, 13.5, 104, 2.5, 136, 4.5, 6.3, 1.4, 24, 136],
        ['Peas, Dried', '1 lb.', 7.9, 20.0, 1367, 4.2, 345, 2.9, 28.7, 18.4, 162, 0],
        ['Lima Beans, Dried', '1 lb.', 8.9, 17.4, 1055, 3.7, 459, 5.1, 26.9, 38.2, 93, 0],
        ['Navy Beans, Dried', '1 lb.', 5.9, 26.9, 1691, 11.4, 792, 0, 38.4, 24.6, 217, 0],
        ['Coffee', '1 lb.', 22.4, 0, 0, 0, 0, 0, 4, 5.1, 50, 0],
        ['Tea', '1/4 lb.', 17.4, 0, 0, 0, 0, 0, 0, 2.3, 42, 0],
        ['Cocoa', '8 oz.', 8.6, 8.7, 237, 3, 72, 0, 2, 11.9, 40, 0],
        ['Chocolate', '8 oz.', 16.2, 8.0, 77, 1.3, 39, 0, 0.9, 3.4, 14, 0],
        ['Sugar', '10 lb.', 51.7, 34.9, 0, 0, 0, 0, 0, 0, 0, 0],
        ['Corn Syrup', '24 oz.', 13.7, 14.7, 0, 0.5, 74, 0, 0, 0, 5, 0],
        ['Molasses', '18 oz.', 13.6, 9.0, 0, 10.3, 244, 0, 1.9, 7.5, 146, 0],
        ['Strawberry Preserves', '1 lb.', 20.5, 6.4, 11, 0.4, 7, 0.2, 0.2, 0.4, 3, 0]];

    # 需要的各种营养素最小需求量
    nutrients = [
        ['Calories (1000s)', 3],
        ['Protein (grams)', 70],
        ['Calcium (grams)', 0.8],
        ['Iron (mg)', 12],
        ['Vitamin A (1000 IU)', 5],
        ['Vitamin B1 (mg)', 1.8],
        ['Vitamin B2 (mg)', 2.7],
        ['Niacin (mg)', 18],
        ['Vitamin C (mg)', 75]]

    # 初始化求解器
    solver = pywraplp.Solver('SolveStigler',
                             pywraplp.Solver.GLOP_LINEAR_PROGRAMMING)
    #
    food = [[]] * len(data)

    # Objective: minimize the sum of (price-normalized) foods.
    objective = solver.Objective()
    for i in range(0, len(data)):
        food[i] = solver.NumVar(0.0, solver.infinity(), data[i][0])     # 定义变量：最小值，最大值，名称
        objective.SetCoefficient(food[i], 1)                            # 定义目标函数的变量的系数

    # 定义求最大值还是最小值
    objective.SetMinimization()

    # 定义约束
    constraints = [0] * len(nutrients)
    for i in range(0, len(nutrients)):
        constraints[i] = solver.Constraint(nutrients[i][1], solver.infinity())
        for j in range(0, len(data)):
            constraints[i].SetCoefficient(food[j], data[j][i + 3])

    # 求解
    status = solver.Solve()

    # 打印结果
    if status == solver.OPTIMAL:
        # Display the amounts (in dollars) to purchase of each food.
        price = 0
        num_nutrients = len(data[i]) - 3
        nutrients = [0] * (len(data[i]) - 3)
        for i in range(0, len(data)):
            price += food[i].solution_value()

            for nutrient in range(0, num_nutrients):
                nutrients[nutrient] += data[i][nutrient + 3] * food[i].solution_value()

            if food[i].solution_value() > 0:
                print("%s = %f" % (data[i][0], food[i].solution_value()))

        print('Optimal annual price: $%.2f' % (365 * price))
    else:  # No optimal solution was found.
        if status == solver.FEASIBLE:
            print('A potentially suboptimal solution was found.')
        else:
            print('The solver could not solve the problem.')


if __name__ == '__main__':
    dt = datetime.datetime
    time_delta = datetime.timedelta
    pyplt = py.offline.plot

    file_name = input("Input file name: ")
    data, check = load_text(file_name)
    if data is not None:
        MinimalJobshopSat(data)


Input file name: D:/pyprj/ginger5/test/math/my_ortools.txt
User time: 0.03s
Wall time: 0.03s
Optimal Schedule Length: 55


In [ ]:
# 排班
import collections
from ortools.sat.python import cp_model
    [  # Job 0
        [(3, 0), (1, 1), (5, 2)],  # task0 有3个可选的加工设备
        [(2, 0), (4, 1), (6, 2)],  # task1 有3个可选的加工设备
        [(2, 0), (3, 1), (1, 2)],  # task2 有3个可选的加工设备
    ],
    [  # Job 1
        [(2, 0), (3, 1), (4, 2)],
        [(1, 0), (5, 1), (4, 2)],
        [(2, 0), (1, 1), (4, 2)],
    ],
    [  # Job 2
        [(2, 0), (1, 1), (4, 2)],
        [(2, 0), (3, 1), (4, 2)],
        [(3, 0), (1, 1), (5, 2)],
    ],
]
 
#机器维护时间
machine_maintenance_time=[
    [(1,3),(6,8)],
    [(0,2)],
    [(1,3),(7,9)]    
]

In [4]:
# 排班
from ortools.sat.python import cp_model
model = cp_model.CpModel()
 
#8名员工
num_employees = 8
 
#排3周的班
num_weeks = 3
 
#每天的班次分为: 休息、早班、中班、晚班
shifts = ['O', 'M', 'A', 'N']
 
#总共排班天数
num_days = num_weeks * 7
 
#班次数量
num_shifts = len(shifts)

#定义排班变量
work = {}
for e in range(num_employees):
    for s in range(num_shifts):
        for d in range(num_days):
            work[e, s, d] = model.NewBoolVar('work%i_%i_%i' % (e, s, d))
 
# 定义线性目标变量组和系数组,在对目标最小化时会使用这些目标变量组和系数组
obj_int_vars = []
obj_int_coeffs = []
obj_bool_vars = []
obj_bool_coeffs = []

# 约束条件：
# 1.  每名员工每天只能上一个班次 (硬)。
# 2. 固定某些员工的班次: 固定某些员工在某天做某个班次 (硬)。
# 3. 员工需求Request: 某些员工希望某天做某个班次 (软)。
# 4. 班次连续天数的约束：某个员工连续上某个班次的天数 (软)。
# 5. 不同班次在每周的合计天数约束 (软)。
# 6. 不同班次之间相互连接的约束 (软)。
# 7. 每周每一天的各班次数量的约束 (软)。

# 每个员工每天只能做一个班次
for e in range(num_employees):
    for d in range(num_days):
        model.Add(sum(work[e, s, d] for s in range(num_shifts)) == 1)


model.addmaxequality
# 固定某些员工的班次: (员工, 班次, 天).
fixed_assignments = [
        (0, 0, 0),
        (1, 0, 0),
        (2, 1, 0),
        (4, 2, 0),
        (5, 2, 0),
        (6, 2, 3),
        (7, 3, 0),
        (0, 1, 1),
        (1, 1, 1),
        (2, 2, 1),
        (3, 2, 1),
        (4, 2, 1),
        (5, 0, 1),
        (6, 0, 1),
        (7, 3, 1),
    ]
# 实现固定班次约束
for e, s, d in fixed_assignments:
    model.Add(work[e, s, d] == 1)


# 员工需求Request: (员工, 班次, 天, 权重)
#负的权重表示某个员工希望某天上某个班次,正的权重表示某个员工不希望某天上某个班次
requests = [
    # 3号员工希望第一个周六休息.
    (3, 0, 5, -2),
    # 5号员工希望第二个周四上晚班.
    (5, 3, 10, -2),
    # 2号员工不希望第一个周五上晚班.
    (2, 3, 4, 4)
# 约束实现:将员工需求加入目标变量中
for e, s, d, w in requests:
    obj_bool_vars.append(work[e, s, d])
    obj_bool_coeffs.append(w)



model.addmaxequality
# 连续天数的约束 :
# (shift, hard_min, soft_min, min_penalty,soft_max, hard_max, max_penalty)
shift_constraints = [
    #可以连续休息1天或者两天,不会被惩罚.
    (0, 1, 1, 0, 2, 2, 0),
    #夜班连续上2天或者3天, 只上1天或者连续上4天都会被惩罚.
    (3, 1, 2, 20, 3, 4, 5),
]
 
#生成一个连续上班天数的固定模式序列
def negated_bounded_span(works, start, length):
    sequence = []
    # 处理序列的左边界 
    if start > 0:
        sequence.append(works[start - 1])
    for i in range(length):
        sequence.append(works[start + i].Not())
    # 处理序列的右边界
    if start + length < len(works):
        sequence.append(works[start + length])
    return sequence
 
 
def add_soft_sequence_constraint(model, works, hard_min, soft_min, min_cost,
                                 soft_max, hard_max, max_cost, prefix):
    cost_literals = []
    cost_coefficients = []
 
    # 禁止连续上小于hard_min天的班
    for length in range(1, hard_min):
        #for start in range(len(works) - length - 1) #源代码中存在的bug
        for start in range(len(works) - length + 1):
            model.AddBoolOr(negated_bounded_span(works, start, length))
 
    # 惩罚连续上班天数落在区间[hard_min,soft_min)中的情况
    if min_cost > 0:
        for length in range(hard_min, soft_min):
            #for start in range(len(works) - length - 1) #源代码中存在的bug
            for start in range(len(works) - length + 1): 
                span = negated_bounded_span(works, start, length)
                name = ': under_span(start=%i, length=%i)' % (start, length)
                lit = model.NewBoolVar(prefix + name)
                span.append(lit)
                model.AddBoolOr(span)
                cost_literals.append(lit)
                # We filter exactly the sequence with a short length.
                # The penalty is proportional to the delta with soft_min.
                cost_coefficients.append(min_cost * (soft_min - length))
 
    # 惩罚连续上班天数落在区间(soft_max ,soft_min]中的情况.
    if max_cost > 0:
        for length in range(soft_max + 1, hard_max + 1):
            #for start in range(len(works) - length - 1) #源代码中存在的bug
            for start in range(len(works) - length + 1): 
                span = negated_bounded_span(works, start, length)
                name = ': over_span(start=%i, length=%i)' % (start, length)
                lit = model.NewBoolVar(prefix + name)
                span.append(lit)
                model.AddBoolOr(span)
                cost_literals.append(lit)
                # Cost paid is max_cost * excess length.
                cost_coefficients.append(max_cost * (length - soft_max))
 
    # 禁止连续上>hard_max天的班
    for start in range(len(works) - hard_max - 1):
        model.AddBoolOr([works[i].Not() for i in range(start, start + hard_max + 1)])
    return cost_literals, cost_coefficients
 
# 班次约束
for ct in shift_constraints:
    shift, hard_min, soft_min, min_cost, soft_max, hard_max, max_cost = ct
    for e in range(num_employees):
        works = [work[e, shift, d] for d in range(num_days)]
        variables, coeffs = add_soft_sequence_constraint(
            model, works, hard_min, soft_min, min_cost, soft_max, hard_max,
            max_cost,
            'shift_constraint(employee %i, shift %i)' % (e, shift))
        obj_bool_vars.extend(variables)
        obj_bool_coeffs.extend(coeffs)



model.addmaxequality
# 班次在每周的天数约束:
#(班次,最小天数(硬), 最小天数(软), 最小惩罚值,最大天数(软), 最大天数(硬), 最大惩罚值)
weekly_sum_constraints = [
    # 每周休息天数的约束.
    (0, 1, 2, 7, 2, 3, 4),
    # 最少1个夜班,最多4个夜班，不上或者超过4个夜班会被惩罚.
    (3, 0, 1, 3, 4, 4, 0),
]
 
 
def add_soft_sum_constraint(model, works, hard_min, soft_min, min_cost,
                            soft_max, hard_max, max_cost, prefix):
    cost_variables = []
    cost_coefficients = []
    sum_var = model.NewIntVar(hard_min, hard_max, '')
    # 计算实际上班天数.
    model.Add(sum_var == sum(works))
 
    # 惩罚合计天数低于soft_min的合计天数.
    if soft_min > hard_min and min_cost > 0:
        delta = model.NewIntVar(-len(works), len(works), '')
        model.Add(delta == soft_min - sum_var)
        # TODO(user): Compare efficiency with only excess >= soft_min - sum_var.
        excess = model.NewIntVar(0, 7, prefix + ': under_sum')
        model.AddMaxEquality(excess, [delta, 0])
        cost_variables.append(excess)
        cost_coefficients.append(min_cost)
 
    # 惩罚合计天数超过soft_max的合计天数.
    if soft_max < hard_max and max_cost > 0:
        delta = model.NewIntVar(-7, 7, '')
        model.Add(delta == sum_var - soft_max)
        excess = model.NewIntVar(0, 7, prefix + ': over_sum')
        model.AddMaxEquality(excess, [delta, 0])
        cost_variables.append(excess)
        cost_coefficients.append(max_cost)
    return cost_variables, cost_coefficients
 
# 对每周的休息班和夜班合计天数的约束
for ct in weekly_sum_constraints:
    shift, hard_min, soft_min, min_cost, soft_max, hard_max, max_cost = ct
    for e in range(num_employees):
        for w in range(num_weeks):
            works = [work[e, shift, d + w * 7] for d in range(7)]
            variables, coeffs = add_soft_sum_constraint(
                model, works, hard_min, soft_min, min_cost, soft_max,
                hard_max, max_cost,
                'weekly_sum_constraint(employee %i, shift %i, week %i)' %
                (e, shift, w))
            obj_int_vars.extend(variables)
            obj_int_coeffs.extend(coeffs)



model.addmaxequality
# 不同班次相互连接的约束:
# (前一个班次, 后一个班次, 惩罚值 (0 意味着 被禁止))
penalized_transitions = [
    # 中班连着晚班,惩罚值是4.
    (2, 3, 4),
    # 夜班连着早班，被禁止.
    (3, 1, 0),
]
 
 
# 惩罚和禁止违规的班次连接
    for e in range(num_employees):
        for d in range(num_days - 1):
            transition = [
                work[e, previous_shift, d].Not(), work[e, next_shift,
                                                       d + 1].Not()
            ]
            if cost == 0:
                model.AddBoolOr(transition)
            else:
                trans_var = model.NewBoolVar(
                    'transition (employee=%i, day=%i)' % (e, d))
                transition.append(trans_var)
                model.AddBoolOr(transition)
                obj_bool_vars.append(trans_var)
                obj_bool_coeffs.append(cost)

True

In [3]:
#"""Creates a shift scheduling problem and solves it."""

from absl import app
from absl import flags

from google.protobuf import text_format
from ortools.sat.python import cp_model

FLAGS = flags.FLAGS
flags.DEFINE_string('output_proto', '',
                    'Output file to write the cp_model proto to.')
flags.DEFINE_string('params', 'max_time_in_seconds:10.0',
                    'Sat solver parameters.')


def negated_bounded_span(works, start, length):
    """Filters an isolated sub-sequence of variables assined to True.

  Extract the span of Boolean variables [start, start + length), negate them,
  and if there is variables to the left/right of this span, surround the span by
  them in non negated form.

  Args:
    works: a list of variables to extract the span from.
    start: the start to the span.
    length: the length of the span.

  Returns:
    a list of variables which conjunction will be false if the sub-list is
    assigned to True, and correctly bounded by variables assigned to False,
    or by the start or end of works.
  """
    sequence = []
    # Left border (start of works, or works[start - 1])
    if start > 0:
        sequence.append(works[start - 1])
    for i in range(length):
        sequence.append(works[start + i].Not())
    # Right border (end of works or works[start + length])
    if start + length < len(works):
        sequence.append(works[start + length])
    return sequence


def add_soft_sequence_constraint(model, works, hard_min, soft_min, min_cost,
                                 soft_max, hard_max, max_cost, prefix):
    """Sequence constraint on true variables with soft and hard bounds.

  This constraint look at every maximal contiguous sequence of variables
  assigned to true. If forbids sequence of length < hard_min or > hard_max.
  Then it creates penalty terms if the length is < soft_min or > soft_max.

  Args:
    model: the sequence constraint is built on this model.
    works: a list of Boolean variables.
    hard_min: any sequence of true variables must have a length of at least
      hard_min.
    soft_min: any sequence should have a length of at least soft_min, or a
      linear penalty on the delta will be added to the objective.
    min_cost: the coefficient of the linear penalty if the length is less than
      soft_min.
    soft_max: any sequence should have a length of at most soft_max, or a linear
      penalty on the delta will be added to the objective.
    hard_max: any sequence of true variables must have a length of at most
      hard_max.
    max_cost: the coefficient of the linear penalty if the length is more than
      soft_max.
    prefix: a base name for penalty literals.

  Returns:
    a tuple (variables_list, coefficient_list) containing the different
    penalties created by the sequence constraint.
  """
    cost_literals = []
    cost_coefficients = []

    # Forbid sequences that are too short.
    for length in range(1, hard_min):
        for start in range(len(works) - length + 1):
            model.AddBoolOr(negated_bounded_span(works, start, length))

    # Penalize sequences that are below the soft limit.
    if min_cost > 0:
        for length in range(hard_min, soft_min):
            for start in range(len(works) - length + 1):
                span = negated_bounded_span(works, start, length)
                name = ': under_span(start=%i, length=%i)' % (start, length)
                lit = model.NewBoolVar(prefix + name)
                span.append(lit)
                model.AddBoolOr(span)
                cost_literals.append(lit)
                # We filter exactly the sequence with a short length.
                # The penalty is proportional to the delta with soft_min.
                cost_coefficients.append(min_cost * (soft_min - length))

    # Penalize sequences that are above the soft limit.
    if max_cost > 0:
        for length in range(soft_max + 1, hard_max + 1):
            for start in range(len(works) - length + 1):
                span = negated_bounded_span(works, start, length)
                name = ': over_span(start=%i, length=%i)' % (start, length)
                lit = model.NewBoolVar(prefix + name)
                span.append(lit)
                model.AddBoolOr(span)
                cost_literals.append(lit)
                # Cost paid is max_cost * excess length.
                cost_coefficients.append(max_cost * (length - soft_max))

    # Just forbid any sequence of true variables with length hard_max + 1
    for start in range(len(works) - hard_max):
        model.AddBoolOr(
            [works[i].Not() for i in range(start, start + hard_max + 1)])
    return cost_literals, cost_coefficients


def add_soft_sum_constraint(model, works, hard_min, soft_min, min_cost,
                            soft_max, hard_max, max_cost, prefix):
    """Sum constraint with soft and hard bounds.

  This constraint counts the variables assigned to true from works.
  If forbids sum < hard_min or > hard_max.
  Then it creates penalty terms if the sum is < soft_min or > soft_max.

  Args:
    model: the sequence constraint is built on this model.
    works: a list of Boolean variables.
    hard_min: any sequence of true variables must have a sum of at least
      hard_min.
    soft_min: any sequence should have a sum of at least soft_min, or a linear
      penalty on the delta will be added to the objective.
    min_cost: the coefficient of the linear penalty if the sum is less than
      soft_min.
    soft_max: any sequence should have a sum of at most soft_max, or a linear
      penalty on the delta will be added to the objective.
    hard_max: any sequence of true variables must have a sum of at most
      hard_max.
    max_cost: the coefficient of the linear penalty if the sum is more than
      soft_max.
    prefix: a base name for penalty variables.

  Returns:
    a tuple (variables_list, coefficient_list) containing the different
    penalties created by the sequence constraint.
  """
    cost_variables = []
    cost_coefficients = []
    sum_var = model.NewIntVar(hard_min, hard_max, '')
    # This adds the hard constraints on the sum.
    model.Add(sum_var == sum(works))

    # Penalize sums below the soft_min target.
    if soft_min > hard_min and min_cost > 0:
        delta = model.NewIntVar(-len(works), len(works), '')
        model.Add(delta == soft_min - sum_var)
        # TODO(user): Compare efficiency with only excess >= soft_min - sum_var.
        excess = model.NewIntVar(0, 7, prefix + ': under_sum')
        model.AddMaxEquality(excess, [delta, 0])
        cost_variables.append(excess)
        cost_coefficients.append(min_cost)

    # Penalize sums above the soft_max target.
    if soft_max < hard_max and max_cost > 0:
        delta = model.NewIntVar(-7, 7, '')
        model.Add(delta == sum_var - soft_max)
        excess = model.NewIntVar(0, 7, prefix + ': over_sum')
        model.AddMaxEquality(excess, [delta, 0])
        cost_variables.append(excess)
        cost_coefficients.append(max_cost)

    return cost_variables, cost_coefficients


def solve_shift_scheduling(params, output_proto):
    """Solves the shift scheduling problem."""
    # Data
    num_employees = 8
    num_weeks = 3
    shifts = ['O', 'M', 'A', 'N']

    # Fixed assignment: (employee, shift, day).
    # This fixes the first 2 days of the schedule.
    fixed_assignments = [
        (0, 0, 0),
        (1, 0, 0),
        (2, 1, 0),
        (3, 1, 0),
        (4, 2, 0),
        (5, 2, 0),
        (6, 2, 3),
        (7, 3, 0),
        (0, 1, 1),
        (1, 1, 1),
        (2, 2, 1),
        (3, 2, 1),
        (4, 2, 1),
        (5, 0, 1),
        (6, 0, 1),
        (7, 3, 1),
    ]

    # Request: (employee, shift, day, weight)
    # A negative weight indicates that the employee desire this assignment.
    requests = [
        # Employee 3 does not want to work on the first Saturday (negative weight
        # for the Off shift).
        (3, 0, 5, -2),
        # Employee 5 wants a night shift on the second Thursday (negative weight).
        (5, 3, 10, -2),
        # Employee 2 does not want a night shift on the first Friday (positive
        # weight).
        (2, 3, 4, 4)
    ]

    # Shift constraints on continuous sequence :
    #     (shift, hard_min, soft_min, min_penalty,
    #             soft_max, hard_max, max_penalty)
    shift_constraints = [
        # One or two consecutive days of rest, this is a hard constraint.
        (0, 1, 1, 0, 2, 2, 0),
        # between 2 and 3 consecutive days of night shifts, 1 and 4 are
        # possible but penalized.
        (3, 1, 2, 20, 3, 4, 5),
    ]

    # Weekly sum constraints on shifts days:
    #     (shift, hard_min, soft_min, min_penalty,
    #             soft_max, hard_max, max_penalty)
    weekly_sum_constraints = [
        # Constraints on rests per week.
        (0, 1, 2, 7, 2, 3, 4),
        # At least 1 night shift per week (penalized). At most 4 (hard).
        (3, 0, 1, 3, 4, 4, 0),
    ]

    # Penalized transitions:
    #     (previous_shift, next_shift, penalty (0 means forbidden))
    penalized_transitions = [
        # Afternoon to night has a penalty of 4.
        (2, 3, 4),
        # Night to morning is forbidden.
        (3, 1, 0),
    ]

    # daily demands for work shifts (morning, afternon, night) for each day
    # of the week starting on Monday.
    weekly_cover_demands = [
        (2, 3, 1),  # Monday
        (2, 3, 1),  # Tuesday
        (2, 2, 2),  # Wednesday
        (2, 3, 1),  # Thursday
        (2, 2, 2),  # Friday
        (1, 2, 3),  # Saturday
        (1, 3, 1),  # Sunday
    ]

    # Penalty for exceeding the cover constraint per shift type.
    excess_cover_penalties = (2, 2, 5)

    num_days = num_weeks * 7
    num_shifts = len(shifts)

    model = cp_model.CpModel()

    work = {}
    for e in range(num_employees):
        for s in range(num_shifts):
            for d in range(num_days):
                work[e, s, d] = model.NewBoolVar('work%i_%i_%i' % (e, s, d))

    # Linear terms of the objective in a minimization context.
    obj_int_vars = []
    obj_int_coeffs = []
    obj_bool_vars = []
    obj_bool_coeffs = []

    # Exactly one shift per day.
    for e in range(num_employees):
        for d in range(num_days):
            model.AddExactlyOne(work[e, s, d] for s in range(num_shifts))

    # Fixed assignments.
    for e, s, d in fixed_assignments:
        model.Add(work[e, s, d] == 1)

    # Employee requests
    for e, s, d, w in requests:
        obj_bool_vars.append(work[e, s, d])
        obj_bool_coeffs.append(w)

    # Shift constraints
    for ct in shift_constraints:
        shift, hard_min, soft_min, min_cost, soft_max, hard_max, max_cost = ct
        for e in range(num_employees):
            works = [work[e, shift, d] for d in range(num_days)]
            variables, coeffs = add_soft_sequence_constraint(
                model, works, hard_min, soft_min, min_cost, soft_max, hard_max,
                max_cost,
                'shift_constraint(employee %i, shift %i)' % (e, shift))
            obj_bool_vars.extend(variables)
            obj_bool_coeffs.extend(coeffs)

    # Weekly sum constraints
    for ct in weekly_sum_constraints:
        shift, hard_min, soft_min, min_cost, soft_max, hard_max, max_cost = ct
        for e in range(num_employees):
            for w in range(num_weeks):
                works = [work[e, shift, d + w * 7] for d in range(7)]
                variables, coeffs = add_soft_sum_constraint(
                    model, works, hard_min, soft_min, min_cost, soft_max,
                    hard_max, max_cost,
                    'weekly_sum_constraint(employee %i, shift %i, week %i)' %
                    (e, shift, w))
                obj_int_vars.extend(variables)
                obj_int_coeffs.extend(coeffs)

    # Penalized transitions
    for previous_shift, next_shift, cost in penalized_transitions:
        for e in range(num_employees):
            for d in range(num_days - 1):
                transition = [
                    work[e, previous_shift, d].Not(), work[e, next_shift,
                                                           d + 1].Not()
                ]
                if cost == 0:
                    model.AddBoolOr(transition)
                else:
                    trans_var = model.NewBoolVar(
                        'transition (employee=%i, day=%i)' % (e, d))
                    transition.append(trans_var)
                    model.AddBoolOr(transition)
                    obj_bool_vars.append(trans_var)
                    obj_bool_coeffs.append(cost)

    # Cover constraints
    for s in range(1, num_shifts):
        for w in range(num_weeks):
            for d in range(7):
                works = [work[e, s, w * 7 + d] for e in range(num_employees)]
                # Ignore Off shift.
                min_demand = weekly_cover_demands[d][s - 1]
                worked = model.NewIntVar(min_demand, num_employees, '')
                model.Add(worked == sum(works))
                over_penalty = excess_cover_penalties[s - 1]
                if over_penalty > 0:
                    name = 'excess_demand(shift=%i, week=%i, day=%i)' % (s, w,
                                                                         d)
                    excess = model.NewIntVar(0, num_employees - min_demand,
                                             name)
                    model.Add(excess == worked - min_demand)
                    obj_int_vars.append(excess)
                    obj_int_coeffs.append(over_penalty)

    # Objective
    model.Minimize(
        sum(obj_bool_vars[i] * obj_bool_coeffs[i]
            for i in range(len(obj_bool_vars))) +
        sum(obj_int_vars[i] * obj_int_coeffs[i]
            for i in range(len(obj_int_vars))))

    if output_proto:
        print('Writing proto to %s' % output_proto)
        with open(output_proto, 'w') as text_file:
            text_file.write(str(model))

    # Solve the model.
    solver = cp_model.CpSolver()
    if params:
        text_format.Parse(params, solver.parameters)
    solution_printer = cp_model.ObjectiveSolutionPrinter()
    status = solver.Solve(model, solution_printer)

    # Print solution.
    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        print()
        header = '          '
        for w in range(num_weeks):
            header += 'M T W T F S S '
        print(header)
        for e in range(num_employees):
            schedule = ''
            for d in range(num_days):
                for s in range(num_shifts):
                    if solver.BooleanValue(work[e, s, d]):
                        schedule += shifts[s] + ' '
            print('worker %i: %s' % (e, schedule))
        print()
        print('Penalties:')
        for i, var in enumerate(obj_bool_vars):
            if solver.BooleanValue(var):
                penalty = obj_bool_coeffs[i]
                if penalty > 0:
                    print('  %s violated, penalty=%i' % (var.Name(), penalty))
                else:
                    print('  %s fulfilled, gain=%i' % (var.Name(), -penalty))

        for i, var in enumerate(obj_int_vars):
            if solver.Value(var) > 0:
                print('  %s violated by %i, linear penalty=%i' %
                      (var.Name(), solver.Value(var), obj_int_coeffs[i]))

    print()
    print('Statistics')
    print('  - status          : %s' % solver.StatusName(status))
    print('  - conflicts       : %i' % solver.NumConflicts())
    print('  - branches        : %i' % solver.NumBranches())
    print('  - wall time       : %f s' % solver.WallTime())


def main(_=None):
    solve_shift_scheduling(FLAGS.params, FLAGS.output_proto)


if __name__ == '__main__':
    app.run(main)

DuplicateFlagError: The flag 'output_proto' is defined twice. First from C:\Users\kim\.virtualenvs\ipython-T2nmkoIa\lib\site-packages\ipykernel_launcher.py, Second from C:\Users\kim\.virtualenvs\ipython-T2nmkoIa\lib\site-packages\ipykernel_launcher.py.  Description from first occurrence: Output file to write the cp_model proto to.

In [4]:
#案例一 ：有三个变量xyz，取值范围0、1或2，约束为x ≠ y。求出可行解：
from ortools.sat.python import cp_model
# 声明模型
model = cp_model.CpModel()
# 创建变量，同时限制变量范围
x = model.NewIntVar(0, 2, 'x')
y = model.NewIntVar(0, 2, 'y')
z = model.NewIntVar(0, 2, 'z')
# 创建约束
model.Add(x != y)
# 调用规划求解器
solver = cp_model.CpSolver()
# 设置规划求解器的最长计算时间
# solver.parameters.max_time_in_seconds = 10
status = solver.Solve(model)
# 判断是否存在有效的解
if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print('x =', solver.Value(x), ', y =',
          solver.Value(y), ', z =', solver.Value(z))
else:
    print('未找到结果')


x = 1 , y = 0 , z = 0


In [21]:
#获取求解器的多个解：

from ortools.sat.python import cp_model


# class MyCpSolver(cp_model.CpSolverSolutionCallback):
#     def __init__(self, variables):
#         cp_model.CpSolverSolutionCallback.__init__(self)
#         self.__variables = variables
#         self.__data = []
#     def on_solution_callback(self):
#         self.__data.append([self.Value(v) for v in self.__variables])
#     def result(self):
#         return self.__data

class MyCpSolver(cp_model.CpSolverSolutionCallback):
    def __init__(self, variables):
        cp_model.CpSolverSolutionCallback.__init__(self)
        self.__variables = variables
        self.__data = []

    def on_solution_callback(self):
        self.__data.append([self.Value(v) for v in self.__variables])
        if len(self.__data) >= 5:
            self.StopSearch()

    def result(self):
        return self.__data
    

# 声明模型
model = cp_model.CpModel()
# 创建变量，同时限制变量范围
x = model.NewIntVar(0, 2, 'x')
y = model.NewIntVar(0, 2, 'y')
z = model.NewIntVar(0, 2, 'z')
# 创建约束
model.Add(x != y)
# 调用规划求解器
solver = cp_model.CpSolver()
myCpSolver = MyCpSolver([x, y, z])
solver.parameters.enumerate_all_solutions = True
status = solver.Solve(model, myCpSolver)
data = myCpSolver.result()
print(solver.StatusName(status), f"共{len(data)}个解：")
for x, y, z in data:
    print(f"x={x}, y={y}, z={z}")


FEASIBLE 共5个解：
x=1, y=0, z=0
x=2, y=0, z=0
x=2, y=0, z=1
x=1, y=0, z=1
x=2, y=1, z=1


In [19]:
myCpSolver.__dict__
cp_model.CpSolverSolutionCallback.__dict__


mappingproxy({'__module__': 'ortools.sat.python.cp_model',
              '__doc__': 'Solution callback.\n\n  This class implements a callback that will be called at each new solution\n  found during search.\n\n  The method OnSolutionCallback() will be called by the solver, and must be\n  implemented. The current solution can be queried using the BooleanValue()\n  and Value() methods.\n\n  It inherits the following methods from its base class:\n\n  * `ObjectiveValue(self)`\n  * `BestObjectiveBound(self)`\n  * `NumBooleans(self)`\n  * `NumConflicts(self)`\n  * `NumBranches(self)`\n  * `WallTime(self)`\n  * `UserTime(self)`\n\n  These methods returns the same information as their counterpart in the\n  `CpSolver` class.\n  ',
              '__init__': <function ortools.sat.python.cp_model.CpSolverSolutionCallback.__init__(self)>,
              'OnSolutionCallback': <function ortools.sat.python.cp_model.CpSolverSolutionCallback.OnSolutionCallback(self)>,
              'BooleanValue': <funct

In [1]:
#由图形组成的等式求解
from ortools.sat.python import cp_model

class MyCpSolver(cp_model.CpSolverSolutionCallback):
    def __init__(self, variables):
        cp_model.CpSolverSolutionCallback.__init__(self)
        self.__variables = variables
        self.__data = []

    def on_solution_callback(self):
        self.__data.append([self.Value(v) for v in self.__variables])

    def result(self):
        return self.__data


model = cp_model.CpModel()
#因为CIFT 是数字的开头，所以不能是0
letters = [model.NewIntVar(int(letter in "CIFT"), 9, letter)
           for letter in "CPISFUNTRE"]
model.AddAllDifferent(letters)
# CP + IS + FUN = TRUE
c, p, i, s, f, u, n, t, r, e = letters
model.Add(c * 10 + p + i * 10 + s + f * 100 + u * 10 +
          n == t * 1000 + r * 100 + u * 10 + e)

solver = cp_model.CpSolver()  #调用规划求解器
myCpSolver = MyCpSolver(letters)
solver.parameters.enumerate_all_solutions = True
status = solver.Solve(model, myCpSolver)
data = myCpSolver.result()
print(f"status={solver.StatusName(status)},conflicts={solver.NumConflicts()},"
      f"branches={solver.NumBranches()},耗时:{solver.WallTime():.4f}s,共{len(data)}个解")
for x, (c, p, i, s, f, u, n, t, r, e) in enumerate(data, 1):
    print(f"第{x:0>2}解: {c}{p} + {i}{s} + {f}{u}{n} = {t}{r}{u}{e}")

status=OPTIMAL,conflicts=179,branches=828,耗时:0.0113s,共72个解
第01解: 64 + 35 + 928 = 1027
第02解: 34 + 65 + 928 = 1027
第03解: 34 + 68 + 925 = 1027
第04解: 32 + 67 + 985 = 1084
第05解: 32 + 65 + 987 = 1084
第06解: 23 + 76 + 985 = 1084
第07解: 23 + 74 + 968 = 1065
第08解: 23 + 75 + 948 = 1046
第09解: 23 + 78 + 945 = 1046
第10解: 23 + 78 + 964 = 1065
第11解: 23 + 75 + 986 = 1084
第12解: 73 + 25 + 986 = 1084
第13解: 73 + 26 + 985 = 1084
第14解: 62 + 37 + 985 = 1084
第15解: 62 + 35 + 987 = 1084
第16解: 73 + 24 + 968 = 1065
第17解: 73 + 28 + 964 = 1065
第18解: 74 + 28 + 963 = 1065
第19解: 78 + 24 + 963 = 1065
第20解: 78 + 23 + 964 = 1065
第21解: 74 + 23 + 968 = 1065
第22解: 76 + 23 + 985 = 1084
第23解: 75 + 23 + 986 = 1084
第24解: 78 + 23 + 945 = 1046
第25解: 78 + 25 + 943 = 1046
第26解: 73 + 25 + 948 = 1046
第27解: 75 + 23 + 948 = 1046
第28解: 73 + 28 + 945 = 1046
第29解: 75 + 28 + 943 = 1046
第30解: 25 + 78 + 943 = 1046
第31解: 46 + 58 + 923 = 1027
第32解: 56 + 48 + 923 = 1027
第33解: 65 + 38 + 924 = 1027
第34解: 65 + 34 + 928 = 1027
第35解: 64 + 38 + 925 = 1

In [11]:
#求解二元一次方程
# x - y = 3
# 3 x - 8 y = 4
from ortools.sat.python import cp_model

model = cp_model.CpModel()

x = model.NewIntVar(cp_model.INT32_MIN, cp_model.INT32_MAX, 'x')
y = model.NewIntVar(cp_model.INT32_MIN, cp_model.INT32_MAX, 'y')
model.Add(x-y == 3)
model.Add(3*x-8*y == 4)
solver = cp_model.CpSolver()
status = solver.Solve(model)
if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print('x =', solver.Value(x), ', y =', solver.Value(y))


x = 4 , y = 1


In [12]:
#线性多项式约束
from ortools.sat.python import cp_model

model = cp_model.CpModel()

x = model.NewIntVar(2, 7, 'x')
y = model.NewIntVar(0, 3, 'y')
model.Add(x + 2*y == 7)
solver = cp_model.CpSolver()
status = solver.Solve(model)
if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print('x =', solver.Value(x), ', y =', solver.Value(y))


x = 7 , y = 0


In [13]:
#八皇后问题

import time
from ortools.sat.python import cp_model


class NQueenSolutionPrinter(cp_model.CpSolverSolutionCallback):

    def __init__(self, queens):
        cp_model.CpSolverSolutionCallback.__init__(self)
        self.__queens = queens
        self.__solution_count = 0

    def solution_count(self):
        return self.__solution_count

    def on_solution_callback(self):
        current_time = time.time()
        self.__solution_count += 1
        print(f" --------- {self.solution_count()} ---------- ")

        for queen in self.__queens:
            for i in range(8):
                if i == self.Value(queen):
                    print(end="Q  ")
                else:
                    print(end="*  ")
            print()


model = cp_model.CpModel()

# 每个皇后必须在不同的行中，记录每行对应的皇后对应的列位置
queens = [model.NewIntVar(0, 7, f'Q_{i}') for i in range(8)]
# 每列最多一个皇后
model.AddAllDifferent(queens)
# 对角线约束
model.AddAllDifferent([queens[i] + i for i in range(8)])
model.AddAllDifferent([queens[i] - i for i in range(8)])

solver = cp_model.CpSolver()
solution_printer = NQueenSolutionPrinter(queens)
solver.parameters.enumerate_all_solutions = True
solver.Solve(model, solution_printer)

statistics=f"""
Statistics
    conflicts : {solver.NumConflicts()}
    branches  : {solver.NumBranches()}
    耗时      ：{solver.WallTime():.4f} s
    共找到    : {solution_printer.solution_count()}个解"""
print(statistics)


 --------- 1 ---------- 
*  *  *  *  Q  *  *  *  
*  *  *  *  *  *  *  Q  
*  *  *  Q  *  *  *  *  
Q  *  *  *  *  *  *  *  
*  *  Q  *  *  *  *  *  
*  *  *  *  *  Q  *  *  
*  Q  *  *  *  *  *  *  
*  *  *  *  *  *  Q  *  
 --------- 2 ---------- 
*  *  *  *  Q  *  *  *  
*  Q  *  *  *  *  *  *  
*  *  *  Q  *  *  *  *  
*  *  *  *  *  *  Q  *  
*  *  Q  *  *  *  *  *  
*  *  *  *  *  *  *  Q  
*  *  *  *  *  Q  *  *  
Q  *  *  *  *  *  *  *  
 --------- 3 ---------- 
*  *  *  *  Q  *  *  *  
*  *  *  *  *  *  Q  *  
*  *  *  Q  *  *  *  *  
Q  *  *  *  *  *  *  *  
*  *  Q  *  *  *  *  *  
*  *  *  *  *  *  *  Q  
*  *  *  *  *  Q  *  *  
*  Q  *  *  *  *  *  *  
 --------- 4 ---------- 
*  *  *  *  Q  *  *  *  
*  Q  *  *  *  *  *  *  
*  *  *  Q  *  *  *  *  
*  *  *  *  *  Q  *  *  
*  *  *  *  *  *  *  Q  
*  *  Q  *  *  *  *  *  
Q  *  *  *  *  *  *  *  
*  *  *  *  *  *  Q  *  
 --------- 5 ---------- 
*  *  *  *  Q  *  *  *  
*  *  *  *  *  *  *  Q  
*  *  *  Q  *  *  *  *  


*  *  *  *  Q  *  *  *  
*  *  *  *  *  *  *  Q  
*  Q  *  *  *  *  *  *  
*  *  *  Q  *  *  *  *  
*  *  *  *  *  Q  *  *  
*  *  Q  *  *  *  *  *  
 --------- 63 ---------- 
*  Q  *  *  *  *  *  *  
*  *  *  *  *  *  Q  *  
*  *  *  *  Q  *  *  *  
*  *  *  *  *  *  *  Q  
Q  *  *  *  *  *  *  *  
*  *  *  Q  *  *  *  *  
*  *  *  *  *  Q  *  *  
*  *  Q  *  *  *  *  *  
 --------- 64 ---------- 
*  *  *  *  *  Q  *  *  
*  *  Q  *  *  *  *  *  
*  *  *  *  Q  *  *  *  
*  *  *  *  *  *  Q  *  
Q  *  *  *  *  *  *  *  
*  *  *  Q  *  *  *  *  
*  Q  *  *  *  *  *  *  
*  *  *  *  *  *  *  Q  
 --------- 65 ---------- 
*  *  Q  *  *  *  *  *  
*  *  *  *  Q  *  *  *  
*  *  *  *  *  *  *  Q  
*  *  *  Q  *  *  *  *  
Q  *  *  *  *  *  *  *  
*  *  *  *  *  *  Q  *  
*  Q  *  *  *  *  *  *  
*  *  *  *  *  Q  *  *  
 --------- 66 ---------- 
*  Q  *  *  *  *  *  *  
*  *  *  *  Q  *  *  *  
*  *  *  *  *  *  Q  *  
Q  *  *  *  *  *  *  *  
*  *  Q  *  *  *  *  *  
*  *  *  *  *  *  *  

In [23]:
#解数独

from ortools.sat.python import cp_model

board = [
    [0, 0, 0, 6, 0, 0, 0, 3, 0],
    [5, 0, 0, 0, 0, 0, 6, 0, 0],
    [0, 9, 0, 0, 0, 5, 0, 0, 0],
    [0, 0, 4, 0, 1, 0, 0, 0, 6],
    [0, 0, 0, 4, 0, 3, 0, 0, 0],
    [8, 0, 0, 0, 9, 0, 5, 0, 0],
    [0, 0, 0, 7, 0, 0, 0, 4, 0],
    [0, 0, 5, 0, 0, 0, 0, 0, 8],
    [0, 3, 0, 0, 0, 8, 0, 0, 0]
]


model = cp_model.CpModel()

# 9x9 整数变量矩阵,数值范围1-9
X = [[model.NewIntVar(1, 9, 'x') for j in range(9)]
     for i in range(9)]
# 每行每个数字最多出现一次
for i in range(9):
    model.AddAllDifferent(X[i])
# 每列每个数字最多出现一次
for j in range(9):
    model.AddAllDifferent([X[i][j] for i in range(9)])
# 每个 3x3 方格每个数字最多出现一次
for i0 in range(3):
    for j0 in range(3):
        constraint = [X[3*i0 + i][3*j0 + j]
                      for i in range(3) for j in range(3)]
        model.AddAllDifferent(constraint)

def solveSudoku(board: list):
    for i in range(9):
        for j in range(9):
            if board[i][j] == 0:
                continue
            model.Add(X[i][j] == board[i][j])

    solver = cp_model.CpSolver()
    status = solver.Solve(model)
    if status != cp_model.OPTIMAL and status != cp_model.FEASIBLE:
        print("无正确的解")
        return
    r = [[solver.Value(X[i][j]) for j in range(9)] for i in range(9)]
    return r

solveSudoku(board)


[[7, 4, 1, 6, 2, 9, 8, 3, 5],
 [5, 8, 2, 3, 4, 1, 6, 9, 7],
 [3, 9, 6, 8, 7, 5, 4, 2, 1],
 [9, 2, 4, 5, 1, 7, 3, 8, 6],
 [6, 5, 7, 4, 8, 3, 2, 1, 9],
 [8, 1, 3, 2, 9, 6, 5, 7, 4],
 [1, 6, 8, 7, 5, 2, 9, 4, 3],
 [2, 7, 5, 9, 3, 4, 1, 6, 8],
 [4, 3, 9, 1, 6, 8, 7, 5, 2]]

In [4]:
import chardet
import time
import datetime
import random
import collections
import plotly as py
import plotly.figure_factory as ff
# Import Python wrapper for or-tools CP-SAT solver.
from ortools.sat.python import cp_model
from ortools.linear_solver import pywraplp  #线性求解


def load_text(file_name):
    try:
        with open(file_name, "rb") as f:
            f_read = f.read()
            f_cha_info = chardet.detect(f_read)
            final_data = f_read.decode(f_cha_info['encoding'])
            return final_data, True
    except FileNotFoundError:
        return str(None), False


def rgb():
    return random.randint(0, 256)


# print(data)
def MinimalJobshopSat(string):
    a = list(map(int, string.split()))
    n, machines_count = a[0], a[1]
    all_machines = range(machines_count)
    jobs_data = []
    job = []
    for i, (j, k) in enumerate(zip(a[2::2], a[3::2])):
        job.append((j, k))
        if (i + 1) % machines_count == 0:
            jobs_data.append(job)
            job = []
    """Minimal jobshop problem."""
    # Create the model.
    model = cp_model.CpModel()

    # Computes horizon dynamically as the sum of all durations.
    horizon = sum(task[1] for job in jobs_data for task in job)

    # Named tuple to store information about created variables.
    task_type = collections.namedtuple('task_type', 'start end interval')
    # Named tuple to manipulate solution information.
    assigned_task_type = collections.namedtuple('assigned_task_type',
                                                'start job index duration')

    # Creates job intervals and add to the corresponding machine lists.
    all_tasks = {}
    machine_to_intervals = collections.defaultdict(list)

    for job_id, job in enumerate(jobs_data):
        for task_id, task in enumerate(job):
            machine = task[0]
            duration = task[1]
            suffix = '_%i_%i' % (job_id, task_id)
            start_var = model.NewIntVar(0, horizon, 'start' + suffix)
            end_var = model.NewIntVar(0, horizon, 'end' + suffix)
            interval_var = model.NewIntervalVar(start_var, duration, end_var,
                                                'interval' + suffix)
            all_tasks[job_id, task_id] = task_type(
                start=start_var, end=end_var, interval=interval_var)
            machine_to_intervals[machine].append(interval_var)

    # Create and add disjunctive constraints.
    for machine in all_machines:
        model.AddNoOverlap(machine_to_intervals[machine])

    # Precedences inside a job.
    for job_id, job in enumerate(jobs_data):
        for task_id in range(len(job) - 1):
            model.Add(all_tasks[job_id, task_id +
                                1].start >= all_tasks[job_id, task_id].end)

    # Makespan objective.
    obj_var = model.NewIntVar(0, horizon, 'makespan')
    model.AddMaxEquality(obj_var, [
        all_tasks[job_id, len(job) - 1].end
        for job_id, job in enumerate(jobs_data)
    ])
    model.Minimize(obj_var)

    # Solve model.
    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL:
        # Create one list of assigned tasks per machine.
        assigned_jobs = collections.defaultdict(list)
        for job_id, job in enumerate(jobs_data):
            for task_id, task in enumerate(job):
                machine = task[0]
                assigned_jobs[machine].append(
                    assigned_task_type(
                        start=solver.Value(all_tasks[job_id, task_id].start),
                        job=job_id,
                        index=task_id,
                        duration=task[1]))

        # Create per machine output lines.
        print('User time: %.2fs' % solver.UserTime())
        print('Wall time: %.2fs' % solver.WallTime())
        print('Optimal Schedule Length: %i' % solver.ObjectiveValue())

        # Draw GanttChart
        df = []
        today = dt.today()
        start_date = dt(today.year, today.month, today.day)
        for machine in all_machines[::-1]:
            # Sort by starting time.
            assigned_jobs[machine].sort()
            for assigned_task in assigned_jobs[machine]:
                # name = 'job_%i_%i' % (assigned_task.job, assigned_task.index)
                start = assigned_task.start
                duration = assigned_task.duration
                df.append(dict(Task="M%s" % (machine + 1), Start=start_date + time_delta(0, start),
                               Finish=start_date + time_delta(0, start + duration),
                               Resource="%s" % (assigned_task.job + 1), complete=assigned_task.job + 1))
        colors = {}
        for i in range(n):
            key = "%s" % (i + 1)
            colors[key] = "rgb(%s, %s, %s)" % (rgb(), rgb(), rgb())
        fig = ff.create_gantt(df, colors=colors, index_col='Resource', group_tasks=True, show_colorbar=True)
        pyplt(fig, filename=r"./GanttChart.html", auto_open=False)

def lp_solver():
    # Commodity, Unit, 1939 price (cents), Calories, Protein (g), Calcium (g), Iron (mg),
    # Vitamin A (IU), Thiamine (mg), Riboflavin (mg), Niacin (mg), Ascorbic Acid (mg)
    data = [
        ['Wheat Flour (Enriched)', '10 lb.', 36, 44.7, 1411, 2, 365, 0, 55.4, 33.3, 441, 0],
        ['Macaroni', '1 lb.', 14.1, 11.6, 418, 0.7, 54, 0, 3.2, 1.9, 68, 0],
        ['Wheat Cereal (Enriched)', '28 oz.', 24.2, 11.8, 377, 14.4, 175, 0, 14.4, 8.8, 114, 0],
        ['Corn Flakes', '8 oz.', 7.1, 11.4, 252, 0.1, 56, 0, 13.5, 2.3, 68, 0],
        ['Corn Meal', '1 lb.', 4.6, 36.0, 897, 1.7, 99, 30.9, 17.4, 7.9, 106, 0],
        ['Hominy Grits', '24 oz.', 8.5, 28.6, 680, 0.8, 80, 0, 10.6, 1.6, 110, 0],
        ['Rice', '1 lb.', 7.5, 21.2, 460, 0.6, 41, 0, 2, 4.8, 60, 0],
        ['Rolled Oats', '1 lb.', 7.1, 25.3, 907, 5.1, 341, 0, 37.1, 8.9, 64, 0],
        ['White Bread (Enriched)', '1 lb.', 7.9, 15.0, 488, 2.5, 115, 0, 13.8, 8.5, 126, 0],
        ['Whole Wheat Bread', '1 lb.', 9.1, 12.2, 484, 2.7, 125, 0, 13.9, 6.4, 160, 0],
        ['Rye Bread', '1 lb.', 9.1, 12.4, 439, 1.1, 82, 0, 9.9, 3, 66, 0],
        ['Pound Cake', '1 lb.', 24.8, 8.0, 130, 0.4, 31, 18.9, 2.8, 3, 17, 0],
        ['Soda Crackers', '1 lb.', 15.1, 12.5, 288, 0.5, 50, 0, 0, 0, 0, 0],
        ['Milk', '1 qt.', 11, 6.1, 310, 10.5, 18, 16.8, 4, 16, 7, 177],
        ['Evaporated Milk (can)', '14.5 oz.', 6.7, 8.4, 422, 15.1, 9, 26, 3, 23.5, 11, 60],
        ['Butter', '1 lb.', 30.8, 10.8, 9, 0.2, 3, 44.2, 0, 0.2, 2, 0],
        ['Oleomargarine', '1 lb.', 16.1, 20.6, 17, 0.6, 6, 55.8, 0.2, 0, 0, 0],
        ['Eggs', '1 doz.', 32.6, 2.9, 238, 1.0, 52, 18.6, 2.8, 6.5, 1, 0],
        ['Cheese (Cheddar)', '1 lb.', 24.2, 7.4, 448, 16.4, 19, 28.1, 0.8, 10.3, 4, 0],
        ['Cream', '1/2 pt.', 14.1, 3.5, 49, 1.7, 3, 16.9, 0.6, 2.5, 0, 17],
        ['Peanut Butter', '1 lb.', 17.9, 15.7, 661, 1.0, 48, 0, 9.6, 8.1, 471, 0],
        ['Mayonnaise', '1/2 pt.', 16.7, 8.6, 18, 0.2, 8, 2.7, 0.4, 0.5, 0, 0],
        ['Crisco', '1 lb.', 20.3, 20.1, 0, 0, 0, 0, 0, 0, 0, 0],
        ['Lard', '1 lb.', 9.8, 41.7, 0, 0, 0, 0.2, 0, 0.5, 5, 0],
        ['Sirloin Steak', '1 lb.', 39.6, 2.9, 166, 0.1, 34, 0.2, 2.1, 2.9, 69, 0],
        ['Round Steak', '1 lb.', 36.4, 2.2, 214, 0.1, 32, 0.4, 2.5, 2.4, 87, 0],
        ['Rib Roast', '1 lb.', 29.2, 3.4, 213, 0.1, 33, 0, 0, 2, 0, 0],
        ['Chuck Roast', '1 lb.', 22.6, 3.6, 309, 0.2, 46, 0.4, 1, 4, 120, 0],
        ['Plate', '1 lb.', 14.6, 8.5, 404, 0.2, 62, 0, 0.9, 0, 0, 0],
        ['Liver (Beef)', '1 lb.', 26.8, 2.2, 333, 0.2, 139, 169.2, 6.4, 50.8, 316, 525],
        ['Leg of Lamb', '1 lb.', 27.6, 3.1, 245, 0.1, 20, 0, 2.8, 3.9, 86, 0],
        ['Lamb Chops (Rib)', '1 lb.', 36.6, 3.3, 140, 0.1, 15, 0, 1.7, 2.7, 54, 0],
        ['Pork Chops', '1 lb.', 30.7, 3.5, 196, 0.2, 30, 0, 17.4, 2.7, 60, 0],
        ['Pork Loin Roast', '1 lb.', 24.2, 4.4, 249, 0.3, 37, 0, 18.2, 3.6, 79, 0],
        ['Bacon', '1 lb.', 25.6, 10.4, 152, 0.2, 23, 0, 1.8, 1.8, 71, 0],
        ['Ham, smoked', '1 lb.', 27.4, 6.7, 212, 0.2, 31, 0, 9.9, 3.3, 50, 0],
        ['Salt Pork', '1 lb.', 16, 18.8, 164, 0.1, 26, 0, 1.4, 1.8, 0, 0],
        ['Roasting Chicken', '1 lb.', 30.3, 1.8, 184, 0.1, 30, 0.1, 0.9, 1.8, 68, 46],
        ['Veal Cutlets', '1 lb.', 42.3, 1.7, 156, 0.1, 24, 0, 1.4, 2.4, 57, 0],
        ['Salmon, Pink (can)', '16 oz.', 13, 5.8, 705, 6.8, 45, 3.5, 1, 4.9, 209, 0],
        ['Apples', '1 lb.', 4.4, 5.8, 27, 0.5, 36, 7.3, 3.6, 2.7, 5, 544],
        ['Bananas', '1 lb.', 6.1, 4.9, 60, 0.4, 30, 17.4, 2.5, 3.5, 28, 498],
        ['Lemons', '1 doz.', 26, 1.0, 21, 0.5, 14, 0, 0.5, 0, 4, 952],
        ['Oranges', '1 doz.', 30.9, 2.2, 40, 1.1, 18, 11.1, 3.6, 1.3, 10, 1998],
        ['Green Beans', '1 lb.', 7.1, 2.4, 138, 3.7, 80, 69, 4.3, 5.8, 37, 862],
        ['Cabbage', '1 lb.', 3.7, 2.6, 125, 4.0, 36, 7.2, 9, 4.5, 26, 5369],
        ['Carrots', '1 bunch', 4.7, 2.7, 73, 2.8, 43, 188.5, 6.1, 4.3, 89, 608],
        ['Celery', '1 stalk', 7.3, 0.9, 51, 3.0, 23, 0.9, 1.4, 1.4, 9, 313],
        ['Lettuce', '1 head', 8.2, 0.4, 27, 1.1, 22, 112.4, 1.8, 3.4, 11, 449],
        ['Onions', '1 lb.', 3.6, 5.8, 166, 3.8, 59, 16.6, 4.7, 5.9, 21, 1184],
        ['Potatoes', '15 lb.', 34, 14.3, 336, 1.8, 118, 6.7, 29.4, 7.1, 198, 2522],
        ['Spinach', '1 lb.', 8.1, 1.1, 106, 0, 138, 918.4, 5.7, 13.8, 33, 2755],
        ['Sweet Potatoes', '1 lb.', 5.1, 9.6, 138, 2.7, 54, 290.7, 8.4, 5.4, 83, 1912],
        ['Peaches (can)', 'No. 2 1/2', 16.8, 3.7, 20, 0.4, 10, 21.5, 0.5, 1, 31, 196],
        ['Pears (can)', 'No. 2 1/2', 20.4, 3.0, 8, 0.3, 8, 0.8, 0.8, 0.8, 5, 81],
        ['Pineapple (can)', 'No. 2 1/2', 21.3, 2.4, 16, 0.4, 8, 2, 2.8, 0.8, 7, 399],
        ['Asparagus (can)', 'No. 2', 27.7, 0.4, 33, 0.3, 12, 16.3, 1.4, 2.1, 17, 272],
        ['Green Beans (can)', 'No. 2', 10, 1.0, 54, 2, 65, 53.9, 1.6, 4.3, 32, 431],
        ['Pork and Beans (can)', '16 oz.', 7.1, 7.5, 364, 4, 134, 3.5, 8.3, 7.7, 56, 0],
        ['Corn (can)', 'No. 2', 10.4, 5.2, 136, 0.2, 16, 12, 1.6, 2.7, 42, 218],
        ['Peas (can)', 'No. 2', 13.8, 2.3, 136, 0.6, 45, 34.9, 4.9, 2.5, 37, 370],
        ['Tomatoes (can)', 'No. 2', 8.6, 1.3, 63, 0.7, 38, 53.2, 3.4, 2.5, 36, 1253],
        ['Tomato Soup (can)', '10 1/2 oz.', 7.6, 1.6, 71, 0.6, 43, 57.9, 3.5, 2.4, 67, 862],
        ['Peaches, Dried', '1 lb.', 15.7, 8.5, 87, 1.7, 173, 86.8, 1.2, 4.3, 55, 57],
        ['Prunes, Dried', '1 lb.', 9, 12.8, 99, 2.5, 154, 85.7, 3.9, 4.3, 65, 257],
        ['Raisins, Dried', '15 oz.', 9.4, 13.5, 104, 2.5, 136, 4.5, 6.3, 1.4, 24, 136],
        ['Peas, Dried', '1 lb.', 7.9, 20.0, 1367, 4.2, 345, 2.9, 28.7, 18.4, 162, 0],
        ['Lima Beans, Dried', '1 lb.', 8.9, 17.4, 1055, 3.7, 459, 5.1, 26.9, 38.2, 93, 0],
        ['Navy Beans, Dried', '1 lb.', 5.9, 26.9, 1691, 11.4, 792, 0, 38.4, 24.6, 217, 0],
        ['Coffee', '1 lb.', 22.4, 0, 0, 0, 0, 0, 4, 5.1, 50, 0],
        ['Tea', '1/4 lb.', 17.4, 0, 0, 0, 0, 0, 0, 2.3, 42, 0],
        ['Cocoa', '8 oz.', 8.6, 8.7, 237, 3, 72, 0, 2, 11.9, 40, 0],
        ['Chocolate', '8 oz.', 16.2, 8.0, 77, 1.3, 39, 0, 0.9, 3.4, 14, 0],
        ['Sugar', '10 lb.', 51.7, 34.9, 0, 0, 0, 0, 0, 0, 0, 0],
        ['Corn Syrup', '24 oz.', 13.7, 14.7, 0, 0.5, 74, 0, 0, 0, 5, 0],
        ['Molasses', '18 oz.', 13.6, 9.0, 0, 10.3, 244, 0, 1.9, 7.5, 146, 0],
        ['Strawberry Preserves', '1 lb.', 20.5, 6.4, 11, 0.4, 7, 0.2, 0.2, 0.4, 3, 0]];

    # 需要的各种营养素最小需求量
    nutrients = [
        ['Calories (1000s)', 3],
        ['Protein (grams)', 70],
        ['Calcium (grams)', 0.8],
        ['Iron (mg)', 12],
        ['Vitamin A (1000 IU)', 5],
        ['Vitamin B1 (mg)', 1.8],
        ['Vitamin B2 (mg)', 2.7],
        ['Niacin (mg)', 18],
        ['Vitamin C (mg)', 75]]

    # 初始化求解器
    solver = pywraplp.Solver('SolveStigler',
                             pywraplp.Solver.GLOP_LINEAR_PROGRAMMING)
    #设置需要求解的变量
    food = [[]] * len(data)

    # Objective: minimize the sum of (price-normalized) foods.
    objective = solver.Objective()
    for i in range(0, len(data)):
        food[i] = solver.NumVar(0.0, solver.infinity(), data[i][0])     # 定义变量：最小值，最大值，名称
        objective.SetCoefficient(food[i], 1)                            # 定义目标函数的变量的系数

    # 定义求最大值还是最小值
    objective.SetMinimization()

    # 定义约束
    constraints = [0] * len(nutrients)
    for i in range(0, len(nutrients)):
        constraints[i] = solver.Constraint(nutrients[i][1], solver.infinity())
        for j in range(0, len(data)):
            constraints[i].SetCoefficient(food[j], data[j][i + 3])

    # 求解
    status = solver.Solve()

    # 打印结果
    if status == solver.OPTIMAL:
        # Display the amounts (in dollars) to purchase of each food.
        price = 0
        num_nutrients = len(data[i]) - 3
        nutrients = [0] * (len(data[i]) - 3)
        for i in range(0, len(data)):
            price += food[i].solution_value()

            for nutrient in range(0, num_nutrients):
                nutrients[nutrient] += data[i][nutrient + 3] * food[i].solution_value()

            if food[i].solution_value() > 0:
                print("%s = %f" % (data[i][0], food[i].solution_value()))

        print('Optimal annual price: $%.2f' % (365 * price))
    else:  # No optimal solution was found.
        if status == solver.FEASIBLE:
            print('A potentially suboptimal solution was found.')
        else:
            print('The solver could not solve the problem.')


if __name__ == '__main__':
    dt = datetime.datetime
    time_delta = datetime.timedelta
    pyplt = py.offline.plot
    
    lp_solver()
    # D:/pyprj/data/tmp_data/my_ortools.txt
#     file_name = input("Input file name: ")
#     data, check = load_text(file_name)
#     if data is not None:
#         MinimalJobshopSat(data)

Wheat Flour (Enriched) = 0.029519
Liver (Beef) = 0.001893
Cabbage = 0.011214
Spinach = 0.005008
Navy Beans, Dried = 0.061029
Optimal annual price: $39.66


In [20]:
# 求解最小化  (x-1)^2 +(y -2.5)^2 
import numpy as np
from scipy.optimize import minimize
def examp_16_4():
    func = lambda x:(x[0] - 1)**2 + (x[1] - 2.5) **2
    x0 = np.asarray([0,0]) #初始解
#     bounds = np.asarray([[0,None],[0,None]])
    cons = ({'type':'ineq','fun':lambda x:x[0] - 2*x[1] + 2},
            {'type':'ineq','fun':lambda x:-x[0] - 2*x[1] + 6},
            {'type':'ineq','fun':lambda x:-x[0] + 2*x[1] + 2},
            {'type':'ineq','fun':lambda x:x[0]}, #bounds转换成constraint
            {'type':'ineq','fun':lambda x:x[1]}
           )
    
    res = minimize(func,x0,
#                    bounds=bounds,
                   constraints=cons
                   ,jac=lambda x:np.asarray([2*(x[0] - 1),2*(x[1] - 2.5)])
#                    ,hess=lambda x:np.asarray([[2,0],[0,2]]) #此样例不需要hess信息
                   ,method='SLSQP'
                  )
    print(res)
if __name__ == '__main__':
    examp_16_4()



     fun: 0.8000000000000037
     jac: array([ 0.8, -1.6])
 message: 'Optimization terminated successfully'
    nfev: 4
     nit: 4
    njev: 4
  status: 0
 success: True
       x: array([1.4, 1.7])


In [ ]:
#订单排产 最优化
from ortools.sat.python import cp_model
# 声明模型
model = cp_model.CpModel()
# 创建变量，同时限制变量范围
x = model.NewIntVar(0, 2, 'x')
y = model.NewIntVar(0, 2, 'y')
z = model.NewIntVar(0, 2, 'z')
# 创建约束
model.Add(x != y)
# 调用规划求解器
solver = cp_model.CpSolver()
# 设置规划求解器的最长计算时间
# solver.parameters.max_time_in_seconds = 10
status = solver.Solve(model)
# 判断是否存在有效的解
if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print('x =', solver.Value(x), ', y =',
          solver.Value(y), ', z =', solver.Value(z))
else:
    print('未找到结果')


In [12]:
import collections
from ortools.sat.python import cp_model
# Create the model.
model = cp_model.CpModel()

jobs_data = [  # task = (machine_id, processing_time).
    [(0, 3), (1, 2), (2, 2)],  # Job0
    [(0, 2), (2, 1), (1, 4)],  # Job1
    [(1, 4), (2, 3)]  # Job2
]

machines_count = 1 + max(task[0] for job in jobs_data for task in job)
all_machines = range(machines_count)
jobs_count = len(jobs_data)
all_jobs = range(jobs_count)

# Compute horizon.
horizon = sum(task[1] for job in jobs_data for task in job)

task_type = collections.namedtuple('task_type', 'start end interval')
assigned_task_type = collections.namedtuple('assigned_task_type','start job index')

my_vars = []
# Create jobs.
all_tasks = {}
for job in all_jobs:
    for task_id, task in enumerate(jobs_data[job]):
        start_var = model.NewIntVar(0, horizon,
                                    'start_%i_%i' % (job, task_id))
        duration = task[1]
        end_var = model.NewIntVar(0, horizon, 'end_%i_%i' % (job, task_id))
        interval_var = model.NewIntervalVar(
            start_var, duration, end_var, 'interval_%i_%i' % (job, task_id))
        all_tasks[job, task_id] = task_type(
            start=start_var, end=end_var, interval=interval_var)
        my_vars.extend([start_var,end_var,interval_var]) 

# Create and add disjunctive constraints.
for machine in all_machines:
    intervals = []
    for job in all_jobs:
        for task_id, task in enumerate(jobs_data[job]):
            if task[0] == machine:
                intervals.append(all_tasks[job, task_id].interval)
    model.AddNoOverlap(intervals)

# Add precedence contraints.
for job in all_jobs:
    for task_id in range(0, len(jobs_data[job]) - 1):
        model.Add(all_tasks[job, task_id +
                            1].start >= all_tasks[job, task_id].end)

# Makespan objective.
obj_var = model.NewIntVar(0, horizon, 'makespan')
model.AddMaxEquality(
    obj_var,
    [all_tasks[(job, len(jobs_data[job]) - 1)].end for job in all_jobs])
model.Minimize(obj_var)

# Solve model.
solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL:
    # Print out makespan.
    print('Optimal Schedule Length: %i' % solver.ObjectiveValue())
    print()

    # Create one list of assigned tasks per machine.
    assigned_jobs = [[] for _ in all_machines]
    for job in all_jobs:
        for task_id, task in enumerate(jobs_data[job]):
            machine = task[0]
            assigned_jobs[machine].append(
                assigned_task_type(
                    start=solver.Value(all_tasks[job, task_id].start),
                    job=job,
                    index=task_id))

    disp_col_width = 10
    sol_line = ''
    sol_line_tasks = ''

    print('Optimal Schedule', '\n')

    for machine in all_machines:
        # Sort by starting time.
        assigned_jobs[machine].sort()
        sol_line += 'Machine ' + str(machine) + ': '
        sol_line_tasks += 'Machine ' + str(machine) + ': '

        for assigned_task in assigned_jobs[machine]:
            name = 'job_%i_%i' % (assigned_task.job, assigned_task.index)
            # Add spaces to output to align columns.
            sol_line_tasks += name + ' ' * (disp_col_width - len(name))
            start = assigned_task.start
            duration = jobs_data[assigned_task.job][assigned_task.index][1]

            sol_tmp = '[%i,%i]' % (start, start + duration)
            # Add spaces to output to align columns.
            sol_line += sol_tmp + ' ' * (disp_col_width - len(sol_tmp))

        sol_line += '\n'
        sol_line_tasks += '\n'

    print(sol_line_tasks)
    print('Task Time Intervals\n')
    print(sol_line)


Optimal Schedule Length: 11

Optimal Schedule 

Machine 0: job_1_0   job_0_0   
Machine 1: job_2_0   job_0_1   job_1_2   
Machine 2: job_1_1   job_2_1   job_0_2   

Task Time Intervals

Machine 0: [0,2]     [2,5]     
Machine 1: [0,4]     [5,7]     [7,11]    
Machine 2: [2,3]     [4,7]     [7,9]     



In [5]:
all_tasks

{(0,
  0): task_type(start=start_0_0(0..21), end=end_0_0(0..21), interval=interval_0_0(start = start_0_0, size = 3, end = end_0_0)),
 (0,
  1): task_type(start=start_0_1(0..21), end=end_0_1(0..21), interval=interval_0_1(start = start_0_1, size = 2, end = end_0_1)),
 (0,
  2): task_type(start=start_0_2(0..21), end=end_0_2(0..21), interval=interval_0_2(start = start_0_2, size = 2, end = end_0_2)),
 (1,
  0): task_type(start=start_1_0(0..21), end=end_1_0(0..21), interval=interval_1_0(start = start_1_0, size = 2, end = end_1_0)),
 (1,
  1): task_type(start=start_1_1(0..21), end=end_1_1(0..21), interval=interval_1_1(start = start_1_1, size = 1, end = end_1_1)),
 (1,
  2): task_type(start=start_1_2(0..21), end=end_1_2(0..21), interval=interval_1_2(start = start_1_2, size = 4, end = end_1_2)),
 (2,
  0): task_type(start=start_2_0(0..21), end=end_2_0(0..21), interval=interval_2_0(start = start_2_0, size = 4, end = end_2_0)),
 (2,
  1): task_type(start=start_2_1(0..21), end=end_2_1(0..21), inte

In [9]:
horizon

21

In [ ]:
  const IntervalVar x = cp_model.NewIntervalVar(cp_model.NewIntVar({0, 20}),
                                                cp_model.NewConstant(5),
                                                cp_model.NewIntVar({0, 20}));
  const IntervalVar y = cp_model.NewIntervalVar(cp_model.NewIntVar({0, 20}),
                                                cp_model.NewConstant(5),
                                                cp_model.NewIntVar({0, 20}));
  const IntVar a = cp_model.NewIntVar({5, 10});
  const IntVar b = cp_model.NewIntVar({5, 10});
  const IntVar c = cp_model.NewIntVar({5, 10});

  CumulativeConstraint cumul = cp_model.AddCumulative(a);
  cumul.AddDemand(x, b);
  cumul.AddDemand(y, c);
    

In [5]:
from collections import namedtuple
obj = namedtuple('testobj', ['A', 'B', 'C', 'D', 'E', 'F'])
b1 = obj('1', '2', '3', '4', '5', '6')
t = ['a1', 'a2', 'a3', 'a4', 'a5', 'a6']
for all t:
    print(i)

SyntaxError: invalid syntax (<ipython-input-5-958c049131a4>, line 5)

In [3]:
import collections
from ortools.sat.python import cp_model
import numpy as np
# Create the model.
model = cp_model.CpModel()

all_steps = [["订单1","工步1"],
    ["订单1","工步2"],
    ["订单1","工步3"],
    ["订单1","工步4"],
    ["订单2","工步2"],
    ["订单2","工步3"],
    ["订单2","工步4"],
    ["订单2","工步5"],
    ["订单3","工步2"],
    ["订单3","工步4"],
    ["订单3","工步5"],
    ["订单4","工步3"],
    ["订单4","工步4"],
    ["订单4","工步6"]]
all_demand = [["订单1","工步1","item1",100],
    ["订单1","工步1","item2",100],
    ["订单1","工步2","item3",200],
    ["订单1","工步3","item4",100],
    ["订单1","工步4","item5",100],
    ["订单1","工步4","item6",100],
    ["订单2","工步2","item3",60],
    ["订单2","工步3","item4",30],
    ["订单2","工步4","item5",30],
    ["订单2","工步4","item6",30],
    ["订单2","工步5","item7",60],
    ["订单3","工步2","item3",40],
    ["订单3","工步4","item5",20],
    ["订单3","工步4","item6",20],
    ["订单3","工步5","item7",40],
    ["订单4","工步3","item4",20],
    ["订单4","工步4","item5",20],
    ["订单4","工步4","item6",20],
    ["订单4","工步6","item7",20]]
all_supply = [["item1",23,30],
    ["item1",24,10],
    ["item1",25,20],
    ["item1",26,10],
    ["item2",23,40],
    ["item3",25,30],
    ["item3",26,20],
    ["item4",31,50],
    ["item5",25,20],
    ["item6",26,30],
    ["item7",31,30],
    ["item1",0,50],
    ["item2",0,60],
    ["item3",0,70],
    ["item4",0,80],
    ["item5",0,90],
    ["item6",0,100],
    ["item7",0,50],
    ["item1",31,1000],
    ["item2",31,1000],
    ["item3",31,1000],
    ["item4",31,1000],
    ["item5",31,1000],
    ["item6",31,1000],
    ["item7",31,1000]]

t_steps = np.array(all_steps)

for i in all_steps:
    i.append(model.NewIntVar(0, 31,f'{i[0]}_{i[1]}'))

    
for i in all_demand:
    for j in all_steps





In [1]:
import numpy as np
all_steps = [["订单1","工步1"],
    ["订单1","工步2"],
    ["订单1","工步3"],
    ["订单1","工步4"],
    ["订单2","工步2"],
    ["订单2","工步3"],
    ["订单2","工步4"],
    ["订单2","工步5"],
    ["订单3","工步2"],
    ["订单3","工步4"],
    ["订单3","工步5"],
    ["订单4","工步3"],
    ["订单4","工步4"],
    ["订单4","工步6"]]
all_demand = [["订单1","工步1","item1",100],
    ["订单1","工步1","item2",100],
    ["订单1","工步2","item3",200],
    ["订单1","工步3","item4",100],
    ["订单1","工步4","item5",100],
    ["订单1","工步4","item6",100],
    ["订单2","工步2","item3",60],
    ["订单2","工步3","item4",30],
    ["订单2","工步4","item5",30],
    ["订单2","工步4","item6",30],
    ["订单2","工步5","item7",60],
    ["订单3","工步2","item3",40],
    ["订单3","工步4","item5",20],
    ["订单3","工步4","item6",20],
    ["订单3","工步5","item7",40],
    ["订单4","工步3","item4",20],
    ["订单4","工步4","item5",20],
    ["订单4","工步4","item6",20],
    ["订单4","工步6","item7",20]]

t = np.array(all_steps)
t[np.where((t[:,0] == "订单1")&(t[:,1] == "工步2")),:]





array([[['订单1', '工步2']]], dtype='<U3')

In [6]:
#vrp示例
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp


def create_data_model():
    """Stores the data for the problem."""
    data = {}
    data["distance_matrix"] = [
        # fmt: off
      [0, 548, 776, 696, 582, 274, 502, 194, 308, 194, 536, 502, 388, 354, 468, 776, 662],
      [548, 0, 684, 308, 194, 502, 730, 354, 696, 742, 1084, 594, 480, 674, 1016, 868, 1210],
      [776, 684, 0, 992, 878, 502, 274, 810, 468, 742, 400, 1278, 1164, 1130, 788, 1552, 754],
      [696, 308, 992, 0, 114, 650, 878, 502, 844, 890, 1232, 514, 628, 822, 1164, 560, 1358],
      [582, 194, 878, 114, 0, 536, 764, 388, 730, 776, 1118, 400, 514, 708, 1050, 674, 1244],
      [274, 502, 502, 650, 536, 0, 228, 308, 194, 240, 582, 776, 662, 628, 514, 1050, 708],
      [502, 730, 274, 878, 764, 228, 0, 536, 194, 468, 354, 1004, 890, 856, 514, 1278, 480],
      [194, 354, 810, 502, 388, 308, 536, 0, 342, 388, 730, 468, 354, 320, 662, 742, 856],
      [308, 696, 468, 844, 730, 194, 194, 342, 0, 274, 388, 810, 696, 662, 320, 1084, 514],
      [194, 742, 742, 890, 776, 240, 468, 388, 274, 0, 342, 536, 422, 388, 274, 810, 468],
      [536, 1084, 400, 1232, 1118, 582, 354, 730, 388, 342, 0, 878, 764, 730, 388, 1152, 354],
      [502, 594, 1278, 514, 400, 776, 1004, 468, 810, 536, 878, 0, 114, 308, 650, 274, 844],
      [388, 480, 1164, 628, 514, 662, 890, 354, 696, 422, 764, 114, 0, 194, 536, 388, 730],
      [354, 674, 1130, 822, 708, 628, 856, 320, 662, 388, 730, 308, 194, 0, 342, 422, 536],
      [468, 1016, 788, 1164, 1050, 514, 514, 662, 320, 274, 388, 650, 536, 342, 0, 764, 194],
      [776, 868, 1552, 560, 674, 1050, 1278, 742, 1084, 810, 1152, 274, 388, 422, 764, 0, 798],
      [662, 1210, 754, 1358, 1244, 708, 480, 856, 514, 468, 354, 844, 730, 536, 194, 798, 0],
        # fmt: on
    ]
    data["pickups_deliveries"] = [
        [1, 6],
        [2, 10],
        [4, 3],
        [5, 9],
        [7, 8],
        [15, 11],
        [13, 12],
        [16, 14],
    ]
    data["num_vehicles"] = 4
    data["depot"] = 0
    return data


def print_solution(data, manager, routing, solution):
    """Prints solution on console."""
    print(f"Objective: {solution.ObjectiveValue()}")
    total_distance = 0
    for vehicle_id in range(data["num_vehicles"]):
        index = routing.Start(vehicle_id)
        plan_output = f"Route for vehicle {vehicle_id}:\n"
        route_distance = 0
        while not routing.IsEnd(index):
            plan_output += f" {manager.IndexToNode(index)} -> "
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(
                previous_index, index, vehicle_id
            )
        plan_output += f"{manager.IndexToNode(index)}\n"
        plan_output += f"Distance of the route: {route_distance}m\n"
        print(plan_output)
        total_distance += route_distance
    print(f"Total Distance of all routes: {total_distance}m")


def main():
    """Entry point of the program."""
    # Instantiate the data problem.
    data = create_data_model()

    # Create the routing index manager.
    manager = pywrapcp.RoutingIndexManager(
        len(data["distance_matrix"]), data["num_vehicles"], data["depot"]
    )

    # Create Routing Model.
    routing = pywrapcp.RoutingModel(manager)


    # Define cost of each arc.
    def distance_callback(from_index, to_index):
        """Returns the manhattan distance between the two nodes."""
        # Convert from routing variable Index to distance matrix NodeIndex.
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return data["distance_matrix"][from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Add Distance constraint.
    dimension_name = "Distance"
    routing.AddDimension(
        transit_callback_index,
        0,  # no slack
        3000,  # vehicle maximum travel distance
        True,  # start cumul to zero
        dimension_name,
    )
    distance_dimension = routing.GetDimensionOrDie(dimension_name)
    distance_dimension.SetGlobalSpanCostCoefficient(100)

    # Define Transportation Requests.
    for request in data["pickups_deliveries"]:
        pickup_index = manager.NodeToIndex(request[0])
        delivery_index = manager.NodeToIndex(request[1])
        routing.AddPickupAndDelivery(pickup_index, delivery_index)
        routing.solver().Add(
            routing.VehicleVar(pickup_index) == routing.VehicleVar(delivery_index)
        )
        routing.solver().Add(
            distance_dimension.CumulVar(pickup_index)
            <= distance_dimension.CumulVar(delivery_index)
        )

    # Setting first solution heuristic.
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
    )

    # Solve the problem.
    solution = routing.SolveWithParameters(search_parameters)

    # Print solution on console.
    if solution:
        print_solution(data, manager, routing, solution)


if __name__ == "__main__":
    main()

Objective: 226116
Route for vehicle 0:
 0 ->  13 ->  15 ->  11 ->  12 -> 0
Distance of the route: 1552m

Route for vehicle 1:
 0 ->  5 ->  2 ->  10 ->  16 ->  14 ->  9 -> 0
Distance of the route: 2192m

Route for vehicle 2:
 0 ->  4 ->  3 -> 0
Distance of the route: 1392m

Route for vehicle 3:
 0 ->  7 ->  1 ->  6 ->  8 -> 0
Distance of the route: 1780m

Total Distance of all routes: 6916m


In [14]:
# P111   4-1 单纯形法求解：
import pandas  as pd
import numpy  as np
#定义线性规划求解函数
def lp_solver(matrix: pd.DataFrame):
    """
    输入线性规划的矩阵，根据单纯法求解线性规划模型
    max cx
    s.t. ax<=b
    矩阵形式是：
            b     x1     x2     x3     x4     x5
    obj   0.0   70.0   30.0    0.0    0.0    0.0
    x3    540    3.0    9.0    1.0    0.0    0.0
    x4    450    5.0    5.0    0.0    1.0    0.0
    x5    720    9.0    3.0    0.0    0.0    1.0
    """
    #检验数是否大于0
    c = matrix.iloc[0,1:]
    while c.max() > 0:
        #选择入基变量，即目标函数系数最大的变量入基
        c = matrix.iloc[0,1:]
        in_x = c.idxmax()
        in_x_v = c[in_x]     #入基变量的系数
        #选择出基变量
        #选择正的最小比值对应的变量出基min(b列/入基变量列)
        b = matrix.iloc[1:,0]
        in_x_a = matrix.iloc[1:][in_x] #选择入基变量对应的列
        out_x = (b/in_x_a).idxmin()  #得到出基变量
        #选择操作
        matrix.loc[out_x,:] = matrix.loc[out_x,:]/matrix.loc[out_x,in_x]
        for idx in matrix.index:
            if idx != out_x:
                matrix.loc[idx,:] = matrix.loc[idx,:]  - matrix.loc[out_x,:] * matrix.loc[idx,in_x]
        #索引替换（入基与出基变量名称替换）
        index= matrix.index.tolist()
        i = index.index(out_x)
        index[i] = in_x
        matrix.index = index
    
    #打印结果
    print("最终的最优单纯形法是：")
    print(matrix)
    print("目标函数值是：", - matrix.iloc[0,0])
    print("最优决策变量是：")
    x_count = (matrix.shape[1] - 1) - (matrix.shape[0] - 1)
    X = matrix.iloc[0,1:].index.tolist()[:x_count]
    for xi in X:
        print(xi,'=',matrix.loc[xi,'b'])
        
#主程序代码
if __name__ == "__main__":
    matrix = pd.DataFrame(np.array([[0,70,30,0,0,0],
                                   [540,3,9,1,0,0],
                                   [450,5,5,0,1,0],
                                   [720,9,3,0,0,1]]),index=['obj','x3','x4','x5'],columns=['b','x1','x2','x3','x4','x5'])
    #调用前面定义的函数求解
    lp_solver(matrix)
    
        


最终的最优单纯形法是：
          b   x1   x2   x3   x4        x5
obj -5700.0  0.0  0.0  0.0 -2.0 -6.666667
x3    180.0  0.0  0.0  1.0 -2.4  1.000000
x2     15.0  0.0  1.0  0.0  0.3 -0.166667
x1     75.0  1.0  0.0  0.0 -0.1  0.166667
目标函数值是： 5700.0
最优决策变量是：
x1 = 75.0
x2 = 15.0


In [16]:
df = pd.DataFrame(np.array([[0,70,30,0,0,0],
                                   [540,3,9,1,0,0],
                                   [450,5,5,0,1,0],
                                   [720,9,3,0,0,1]]),index=['obj','x3','x4','x5'],columns=['b','x1','x2','x3','x4','x5'])

In [21]:
c = df.iloc[0,1:]
c.idxmax()

'x1'